In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, monotonically_increasing_id,
    to_date, date_format, year, quarter, month, dayofmonth,
    dayofweek, weekofyear, when, min as spark_min, max as spark_max
)
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import timedelta

# Configuration
source_schema = "silver_olist"
source_object = "silver_orders"

target_schema = "gold_olist"
target_object = "DimDate"
surrogate_key = "DateKey"

key_cols_list = ["DateKey"]
cdc_col = "full_date"
backdated_col = ""

# Get date range from source orders
df_orders = spark.table(f"{source_schema}.{source_object}")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

available_date_columns = [
    c for c in date_columns if c in df_orders.columns
]

date_expressions = [
    F.to_date(F.col(c)).alias(c)
    for c in available_date_columns
]

df_order_dates = df_orders.select(*date_expressions)

date_range = df_order_dates.select(
    F.least(*[F.min(c) for c in available_date_columns]).alias("min_date"),
    F.greatest(*[F.max(c) for c in available_date_columns]).alias("max_date")
).first()

min_date = date_range["min_date"]
max_date = date_range["max_date"]

if min_date is None or max_date is None:
    raise ValueError("No valid order dates found in silver_orders")

# Last load
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    last_load = spark.sql(
        f"SELECT MAX({cdc_col}) FROM {target_schema}.{target_object}"
    ).collect()[0][0]
else:
    last_load = "1900-01-01"

print(last_load)

# df_source: generate calendar dates
df_calendar = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE('{min_date}'),
            DATE('{max_date}'),
            INTERVAL 1 DAY
        )
    ) AS full_date
""")

df_source = (
    df_calendar
    .withColumn("DateKey", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("year_month", date_format(col("full_date"), "yyyy-MM"))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn(
        "is_weekend",
        when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False)
    )
    .withColumn("created_at", current_timestamp())
    .withColumn("updated_at", current_timestamp())
)

# df_target
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    df_target = spark.sql(f"""
        SELECT DateKey, created_at, updated_at
        FROM {target_schema}.{target_object}
    """)
else:
    df_target = (
        df_source.limit(0)
        .select("DateKey")
        .withColumn("created_at", F.lit(None).cast("timestamp"))
        .withColumn("updated_at", F.lit(None).cast("timestamp"))
    )

# df_join
df_source.createOrReplaceTempView("src")
df_target.createOrReplaceTempView("trg")

df_join = spark.sql("""
    SELECT
        src.*,
        trg.created_at AS target_created_at,
        trg.updated_at AS target_updated_at
    FROM src
    LEFT JOIN trg
        ON src.DateKey = trg.DateKey
""")

# df_new and df_old
df_new = df_join.filter(col("target_created_at").isNull())

df_old = df_join.filter(col("target_created_at").isNotNull())

# df_new_enriched
df_new_enriched = (
    df_new
    .drop("target_created_at", "target_updated_at")
    .withColumn("created_at", current_timestamp())
    .withColumn("updated_at", current_timestamp())
)

# df_old_enriched
df_old_enriched = (
    df_old
    .drop("created_at", "updated_at")
    .withColumnRenamed("target_created_at", "created_at")
    .withColumnRenamed("target_updated_at", "updated_at")
)

# df_union
df_union = df_old_enriched.unionByName(df_new_enriched)

# Delta merge
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(
        spark, f"{target_schema}.{target_object}"
    )

    (
        dlt_obj.alias("trg")
        .merge(
            df_union.alias("src"),
            "src.DateKey = trg.DateKey"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        df_union.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{target_schema}.{target_object}")
    )